# 第19章 分类

这个 notebook 用一个小型 SDSS 风格教学数据集，把分类任务拆成清楚的几个层次：规则 baseline、最小 kNN 分类器、错误分布和样本级诊断。

## 学习目标

- 理解分类输出的是离散标签
- 对比规则方法和数据驱动分类器
- 读懂准确率、混淆矩阵和按类别召回率
- 查看错误样本为何会被混淆
- 建立“边界来自样本分布”的直觉

In [ ]:
from __future__ import annotations

import csv
import math
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/object_classification_demo.csv')
with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    for key in ('u_g', 'g_r', 'r_i', 'i_z'):
        row[key] = float(row[key])

classes = sorted({row['object_class'] for row in rows})
print('loaded rows:', len(rows))
print('classes:', classes)
print('class counts:', dict(Counter(row['object_class'] for row in rows)))
print('Python version:', platform.python_version())


In [ ]:
print('feature ranges by class:')
for label in classes:
    subset = [row for row in rows if row['object_class'] == label]
    print(
        label,
        {
            'u_g': (round(min(row['u_g'] for row in subset), 3), round(max(row['u_g'] for row in subset), 3)),
            'g_r': (round(min(row['g_r'] for row in subset), 3), round(max(row['g_r'] for row in subset), 3)),
            'r_i': (round(min(row['r_i'] for row in subset), 3), round(max(row['r_i'] for row in subset), 3)),
        },
    )


## 一个固定的教学切分

为了让每一类在训练集和测试集中都出现，我们继续使用固定索引的教学切分。

In [ ]:
test_indices = {1, 4, 5, 8, 11, 14}
train_rows = [row for index, row in enumerate(rows) if index not in test_indices]
test_rows = [row for index, row in enumerate(rows) if index in test_indices]

print('train:', len(train_rows), 'test:', len(test_rows))
print('test objects:', [row['object_id'] for row in test_rows])


## 规则 baseline

先写一个简单颜色阈值分类器。它未必最优，但它给我们一个可解释的出发点。

In [ ]:
def classify_rule(row):
    if row['u_g'] < 0.5 and row['g_r'] < 0.1:
        return 'quasar'
    if row['g_r'] > 0.7 and row['r_i'] > 0.3:
        return 'galaxy'
    return 'star'

rule_rows = [
    (row['object_id'], row['object_class'], classify_rule(row))
    for row in test_rows
]
rule_rows


## 一个最小 kNN 分类器

这里使用三个颜色特征，构造一个不依赖第三方机器学习库的 kNN 实现。

In [ ]:
def feature_vector(row):
    return (row['u_g'], row['g_r'], row['r_i'])

def euclidean_distance(a_value, b_value):
    return math.sqrt(sum((ax - bx) ** 2 for ax, bx in zip(a_value, b_value)))

def classify_knn(target_row, training_rows, k_neighbors=3):
    target_features = feature_vector(target_row)
    distances = []
    for row in training_rows:
        distances.append((euclidean_distance(target_features, feature_vector(row)), row['object_class'], row['object_id']))
    distances.sort(key=lambda item: item[0])
    votes = {}
    for _, label, _ in distances[:k_neighbors]:
        votes[label] = votes.get(label, 0) + 1
    predicted = max(sorted(votes), key=lambda label: votes[label])
    return predicted, distances[:k_neighbors]

knn_rows = []
for row in test_rows:
    predicted, neighbors = classify_knn(row, train_rows, k_neighbors=3)
    knn_rows.append((row['object_id'], row['object_class'], predicted, neighbors))

[(item[0], item[1], item[2]) for item in knn_rows]


## 指标：准确率、混淆矩阵、按类别召回率

分类任务里，单看准确率不够。我们还需要知道模型具体把哪些类别混淆了。

In [ ]:
truths = [row['object_class'] for row in test_rows]
rule_predictions = [item[2] for item in rule_rows]
knn_predictions = [item[2] for item in knn_rows]

def accuracy_score(y_true, y_pred):
    return sum(1 for truth, pred in zip(y_true, y_pred) if truth == pred) / len(y_true)

def confusion_matrix(y_true, y_pred, labels):
    matrix = {truth: {pred: 0 for pred in labels} for truth in labels}
    for truth, pred in zip(y_true, y_pred):
        matrix[truth][pred] += 1
    return matrix

def per_class_recall(matrix, labels):
    recalls = {}
    for label in labels:
        total = sum(matrix[label].values())
        recalls[label] = matrix[label][label] / total if total else 0.0
    return recalls

rule_accuracy = accuracy_score(truths, rule_predictions)
knn_accuracy = accuracy_score(truths, knn_predictions)
rule_matrix = confusion_matrix(truths, rule_predictions, classes)
knn_matrix = confusion_matrix(truths, knn_predictions, classes)

print('rule accuracy =', round(rule_accuracy, 3))
print('knn accuracy =', round(knn_accuracy, 3))
print('rule recall =', per_class_recall(rule_matrix, classes))
print('knn recall =', per_class_recall(knn_matrix, classes))
print('rule confusion matrix =', rule_matrix)
print('knn confusion matrix =', knn_matrix)


In [ ]:
misclassified = [item for item in knn_rows if item[1] != item[2]]
print('misclassified rows:')
for object_id, truth, pred, neighbors in misclassified:
    print(object_id, truth, '->', pred)
    print('  nearest neighbors:', [(round(distance, 3), label, neighbor_id) for distance, label, neighbor_id in neighbors])


## 如果安装了 scikit-learn

下面这个单元是可选的。它把最小教学实现与逻辑回归、决策树这些标准工具连起来。

In [ ]:
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
except ModuleNotFoundError:
    print('scikit-learn 未安装；已跳过标准库分类示例。')
else:
    x_train = [list(feature_vector(row)) for row in train_rows]
    y_train = [row['object_class'] for row in train_rows]
    x_test = [list(feature_vector(row)) for row in test_rows]

    logistic_model = LogisticRegression(max_iter=1000)
    logistic_model.fit(x_train, y_train)
    logistic_predictions = logistic_model.predict(x_test)
    print('logistic accuracy =', round(accuracy_score(truths, logistic_predictions), 3))

    tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
    tree_model.fit(x_train, y_train)
    tree_predictions = tree_model.predict(x_test)
    print('decision-tree accuracy =', round(accuracy_score(truths, tree_predictions), 3))


## 小结

这个教学例子说明：分类任务不是只看最后标签，而是要看决策边界如何形成、哪些样本靠近边界、以及错误究竟发生在什么对象上。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'rule_accuracy': round(rule_accuracy, 3),
    'knn_accuracy': round(knn_accuracy, 3),
    'misclassified_count': len(misclassified),
    'python_version': platform.python_version(),
}

print('Classification snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
